# 02 · Training

All models are trained with `python -m ml.src.train --config ml/configs/<name>.json` (AdamW, class-weighted cross-entropy, cosine schedule, early stopping on validation macro-F1, seed 42). This notebook loads the recorded histories.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'ml' / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
DATA_DIR = os.environ.get('WASTEWISE_DATA_DIR', str(ROOT / 'ml/datasets/processed'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 160)

In [ ]:
cfgs = {p.stem: json.load(open(p)) for p in sorted(Path('ml/configs').glob('*.json'))}
pd.DataFrame(cfgs).T

In [ ]:
hist = {}
for d in sorted(Path('ml/models/experiments').glob('*/training_history.json')):
    hist[d.parent.name] = pd.DataFrame(json.load(open(d))['history'])
list(hist)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name, h in hist.items():
    axes[0].plot(h['epoch'], h['train_loss'], '--', label=f'{name} train'); axes[0].plot(h['epoch'], h['val_loss'], label=f'{name} val')
    axes[1].plot(h['epoch'], h['val_acc'], label=name); axes[2].plot(h['epoch'], h['val_f1_macro'], label=name)
for ax, t in zip(axes, ['loss', 'validation accuracy', 'validation macro-F1']): ax.set_title(t); ax.set_xlabel('epoch'); ax.legend(fontsize=7)
plt.tight_layout()

In [ ]:
summ = {d.parent.name: json.load(open(d)) for d in sorted(Path('ml/models/experiments').glob('*/train_summary.json'))}
pd.json_normalize(list(summ.values())).set_index('experiment_id').drop(columns=['checkpoint'])